#Chuẩn bị dataset


*   Không cần chạy vì có sẵn dataset ở drive rồi





In [ ]:
import os
import sys
import io
import cv2
import json
import numpy as np
import shutil
from pycocotools.coco import COCO
from tqdm import tqdm
from google.colab import drive
# 1. CẤU HÌNH THÔNG SỐ VÀ ĐƯỜNG DẪN
TRAIN_LIMIT = None
VAL_LIMIT = None      # None = lấy toàn bộ val2017
DRIVE_WORKSPACE = "/content/gdrive/MyDrive/PowerPaint_KhoaLuan"
RAW_ZIP_PATH = "/content/gdrive/MyDrive/COCO_Raw_Data"
CLEAN_DATASET = "/content/coco_full_clean"
TEMP_EXTRACT = "/content/coco_temp"
COCO_URLS = {
    "train2017.zip": "http://images.cocodataset.org/zips/train2017.zip",
    "val2017.zip": "http://images.cocodataset.org/zips/val2017.zip",
    "annotations_trainval2017.zip": "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
}
SPLIT_SPECS = [
    ("train", "train2017", f"{TEMP_EXTRACT}/train2017", TRAIN_LIMIT),
    ("val", "val2017", f"{TEMP_EXTRACT}/val2017", VAL_LIMIT),
]
# 2. KHỞI TẠO MÔI TRƯỜNG
drive.mount('/content/gdrive')
os.makedirs(DRIVE_WORKSPACE, exist_ok=True)
os.makedirs(RAW_ZIP_PATH, exist_ok=True)
os.makedirs(CLEAN_DATASET, exist_ok=True)
# 3. TẢI DATASET GỐC VÀ LƯU DRIVE
print("Download Dataset COCO 2017")
for file_name, url in COCO_URLS.items():
    dst_drive = os.path.join(RAW_ZIP_PATH, file_name)
    if not os.path.exists(dst_drive):
        print(f"Đang tải {file_name}")
        !wget -q -c {url} -O /content/{file_name}
        print(f"Đang lưu {file_name} vào Drive")
        !cp /content/{file_name} {dst_drive}
    else:
        if not os.path.exists(f"/content/{file_name}"):
            !cp {dst_drive} /content/
# 4. GIẢI NÉN DATASET GỐC
if not os.path.exists(TEMP_EXTRACT):
    print("\nĐang giải nén dataset COCO gốc")
    !unzip -q /content/annotations_trainval2017.zip -d {TEMP_EXTRACT}
    !unzip -q /content/train2017.zip -d {TEMP_EXTRACT}
    !unzip -q /content/val2017.zip -d {TEMP_EXTRACT}
    !rm /content/train2017.zip /content/val2017.zip /content/annotations_trainval2017.zip

# 5. HÀM TRÍCH XUẤT CHUẨN HÓA (TRAIN/VAL)
def process_coco_split(split_name, annotation_mode, img_dir, limit=None):
    print(f"\nĐang xử lý tập [{split_name.upper()}]")
    inst_file = f'{TEMP_EXTRACT}/annotations/instances_{annotation_mode}.json'
    capt_file = f'{TEMP_EXTRACT}/annotations/captions_{annotation_mode}.json'
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    coco_mask = COCO(inst_file)
    coco_text = COCO(capt_file)
    sys.stdout = old_stdout

    split_root = os.path.join(CLEAN_DATASET, split_name)
    if os.path.exists(split_root):
        shutil.rmtree(split_root)

    img_out = os.path.join(split_root, "images")
    mask_out = os.path.join(split_root, "masks")
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(mask_out, exist_ok=True)

    img_ids = coco_mask.getImgIds()
    img_ids.sort()
    img_ids_target = img_ids if limit is None else img_ids[:limit]

    metadata = []
    for img_id in tqdm(img_ids_target, desc=f"Lọc & Tạo Mask {split_name}"):
        img_info = coco_mask.loadImgs(img_id)[0]
        file_name = img_info['file_name']
        src_img_path = os.path.join(img_dir, file_name)
        if not os.path.exists(src_img_path):
            continue

        ann_ids = coco_mask.getAnnIds(imgIds=img_id)
        anns = coco_mask.loadAnns(ann_ids)
        mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)
        for ann in anns:
            mask = np.maximum(mask, coco_mask.annToMask(ann) * 255)
        if np.max(mask) == 0:
            continue

        ann_text_ids = coco_text.getAnnIds(imgIds=img_id)
        text_anns = coco_text.loadAnns(ann_text_ids)
        caption = text_anns[0]['caption'] if len(text_anns) > 0 else ""
        if not caption:
            continue

        dst_img_path = os.path.join(img_out, file_name)
        dst_mask_path = os.path.join(mask_out, file_name.replace('.jpg', '.png'))
        shutil.copy(src_img_path, dst_img_path)
        cv2.imwrite(dst_mask_path, mask)
        metadata.append({
            "image_path": f"{split_name}/images/{file_name}",
            "mask_path": f"{split_name}/masks/{file_name.replace('.jpg', '.png')}",
            "caption": caption.strip()
        })

    with open(os.path.join(split_root, "metadata.json"), "w") as f:
        json.dump(metadata, f, indent=4)
    print(f"Tập {split_name.upper()} có {len(metadata)} mẫu hợp lệ")

# THỰC THI CHO 2 TẬP
for split_name, annotation_mode, img_dir, limit in SPLIT_SPECS:
    process_coco_split(split_name, annotation_mode, img_dir, limit)

# 6. NÉN TỪNG SPLIT VÀ ĐẨY LÊN DRIVE
for split_name, _, _, _ in SPLIT_SPECS:
    zip_name = f"coco_clean_{split_name}.zip"
    split_root = os.path.join(CLEAN_DATASET, split_name)
    if os.path.exists(f"/content/{zip_name}"):
        os.remove(f"/content/{zip_name}")
    %cd {CLEAN_DATASET}
    !zip -r -q /content/{zip_name} {split_name}
    %cd /content
    print(f"Đưa {zip_name} lên Drive")
    !cp /content/{zip_name} {RAW_ZIP_PATH}/

print("Done")


#Cài đặt thư viện

In [ ]:
# KHỞI TẠO MÔI TRƯỜNG POWERPAINT
import os
import sys
from google.colab import drive
# Mount Google Drive
os.chdir("/content")
drive.mount("/content/gdrive")
# Đường dẫn
DRIVE_WORKSPACE = "/content/gdrive/MyDrive/Train_PowerPaint"
REPO_DIR = os.path.join(DRIVE_WORKSPACE, "PowerPaint")
# Tạo thư mục cần thiết
os.makedirs(DRIVE_WORKSPACE, exist_ok=True)
# Cài Python 3.10 + pip
!apt-get update -y > /dev/null
!apt-get install python3.10 python3.10-distutils -y > /dev/null
if not os.path.exists("/content/get-pip.py"):
    !wget -q https://bootstrap.pypa.io/get-pip.py
!python3.10 /content/get-pip.py > /dev/null
# Cài requirements
os.chdir(REPO_DIR)
!python3.10 -m pip install -r requirements/requirements.txt \
    --extra-index-url https://download.pytorch.org/whl/cu118
print("Khởi tạo môi trường xong")

Mounted at /content/gdrive
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 44.2 MB/s  0:00:07
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 144.9 MB/s  0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of xformers to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 905.3/905.3 MB 42.5 MB/s  0:00:11
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 95.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 70.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

#Train

In [ ]:
import os
import sys
import glob
import json
import torch
import gc
import yaml
from pathlib import Path
from google.colab import drive
#THÔNG SỐ HYPERPARAMETERS
COCO_LIMIT = None               # Số lượng ảnh muốn lấy từ tập TRAIN
#KHỞI TẠO ĐƯỜNG DẪN & MÔI TRƯỜNG
drive.mount('/content/gdrive')
DRIVE_WORKSPACE = "/content/gdrive/MyDrive/Train_PowerPaint"
REPO_DIR = os.path.join(DRIVE_WORKSPACE, "PowerPaint")
OUTPUT_DIR = os.path.join(DRIVE_WORKSPACE, "output_models")
POWERPAINT_RELEASE_ROOT = os.path.join(REPO_DIR, "checkpoints", "ppt-v2")
POWERPAINT_BRUSHNET = os.path.join(POWERPAINT_RELEASE_ROOT, "PowerPaint_Brushnet")
POWERPAINT_BASE_MODEL = os.path.join(POWERPAINT_RELEASE_ROOT, "realisticVisionV60B1_v51VAE")
TRAIN_CONFIG_PATH = os.path.join(REPO_DIR, "configs", "config_coco.yaml")
os.makedirs(DRIVE_WORKSPACE, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
#Khai báo đường dẫn Repo vào hệ thống
sys.path.insert(0, REPO_DIR)
# Lọc log cảnh báo rác
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PYTHONWARNINGS"] = "ignore"
#CHUẨN BỊ DỮ LIỆU
print("\nPrepare Data Train")
TRAIN_ZIP_PATH = "/content/gdrive/MyDrive/COCO_Raw_Data/coco_clean_train.zip"
BASE_DATA_DIR = "/content/coco_clean_splits"
DATASET_DIR = os.path.join(BASE_DATA_DIR, "train")
os.makedirs(BASE_DATA_DIR, exist_ok=True)
if not os.path.exists(os.path.join(DATASET_DIR, "metadata.json")):
    !cp "{TRAIN_ZIP_PATH}" /content/train_data.zip
    !unzip -q /content/train_data.zip -d {BASE_DATA_DIR}
    !rm /content/train_data.zip
with open(os.path.join(DATASET_DIR, "metadata.json"), "r") as f:
    full_metadata = json.load(f)
actual_metadata = full_metadata if COCO_LIMIT is None else full_metadata[:COCO_LIMIT]
for item in actual_metadata:
    img_filename = os.path.basename(item["image_path"])
    mask_filename = os.path.basename(item["mask_path"])
    item["image_path"] = os.path.join(DATASET_DIR, "images", img_filename)
    item["mask_path"] = os.path.join(DATASET_DIR, "masks", mask_filename)
with open(os.path.join(DATASET_DIR, "metadata_run.json"), "w") as f:
    json.dump(actual_metadata, f, indent=4)
actual_count = len(actual_metadata)
print(f"Chọn {actual_count} mẫu Data hợp lệ cho quá trình train.")
with open(TRAIN_CONFIG_PATH, "r", encoding="utf-8") as f:
    train_cfg = yaml.safe_load(f)
if not os.path.isdir(POWERPAINT_BRUSHNET):
    raise FileNotFoundError(f"Khong tim thay PowerPaint_Brushnet tai: {POWERPAINT_BRUSHNET}")
if not os.path.isdir(POWERPAINT_BASE_MODEL):
    raise FileNotFoundError(f"Khong tim thay base model realisticVision tai: {POWERPAINT_BASE_MODEL}")
train_cfg["pretrained_model_name_or_path"] = POWERPAINT_BASE_MODEL
train_cfg["powerpaint_model_name_or_path"] = POWERPAINT_BRUSHNET
train_cfg["base_model_name_or_path"] = POWERPAINT_BASE_MODEL
train_cfg["output_dir"] = OUTPUT_DIR
train_cfg["train_data"]["datasets"][0]["data_root"] = DATASET_DIR
with open(TRAIN_CONFIG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(train_cfg, f, sort_keys=False)
print("Train config synced to local PowerPaint release:")
print({
    "pretrained_model_name_or_path": train_cfg["pretrained_model_name_or_path"],
    "powerpaint_model_name_or_path": train_cfg["powerpaint_model_name_or_path"],
    "base_model_name_or_path": train_cfg["base_model_name_or_path"],
    "output_dir": train_cfg["output_dir"],
    "train_data_root": train_cfg["train_data"]["datasets"][0]["data_root"],
})
#KHỞI CHẠY TRAINING
#Dọn dẹp VRAM trước khi chạy
torch.cuda.empty_cache(); gc.collect()
#Logic tìm và chạy tiếp từ file lưu (Resume)
ckpt_list = glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*"))
resume_arg = ""

print("\nKiểm tra Checkpoint...")
if ckpt_list:
    latest_ckpt = max(ckpt_list, key=os.path.getctime)
    resume_arg = f'--resume_from_checkpoint="{os.path.basename(latest_ckpt)}"'
    print(f"tìm thấy bản lưu{latest_ckpt}")
else:
    print("Không tìm thấy bản lưu")
os.chdir(REPO_DIR)
!python3.10 train_ppt2_bn.py --config "configs/config_coco.yaml" {resume_arg}
    # --max_train_steps=500
    # --checkpointing_steps=250


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).

 Prepare Data Train
Chọn 117266 mẫu Data hợp lệ cho quá trình train.
Train config synced to local PowerPaint release:
{'pretrained_model_name_or_path': '/content/gdrive/MyDrive/Train_PowerPaint/PowerPaint/checkpoints/ppt-v2/realisticVisionV60B1_v51VAE', 'powerpaint_model_name_or_path': '/content/gdrive/MyDrive/Train_PowerPaint/PowerPaint/checkpoints/ppt-v2/PowerPaint_Brushnet', 'base_model_name_or_path': '/content/gdrive/MyDrive/Train_PowerPaint/PowerPaint/checkpoints/ppt-v2/realisticVisionV60B1_v51VAE', 'output_dir': '/content/gdrive/MyDrive/Train_PowerPaint/output_models'}

Kiểm tra Checkpoint...
Không tìm thấy bản lưu
06/26/2026 12:34:02 - INFO - __main__ - [RANK 0] Distributed environment: NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

{'brushnet'} was not found in config. Values will be initialize